# Embedding bộ chunk đã lọc — Colab (GPU)
Chạy embedding cho 12.910 chunk đã lọc (v7.3.2) bằng 2 model để so sánh: **bge-m3** (tạm chốt) và **multilingual-e5-large** (nhẹ hơn, model còn lại trong vòng chung kết). Vector lưu ra file .npz để dùng ở bước 5 (đánh giá truy xuất).

**Làm theo đúng thứ tự:**
1. Mở notebook này trên Google Colab.
2. Runtime → Change runtime type → Hardware accelerator: **T4 GPU** → Save.
3. Chạy từng ô theo thứ tự từ trên xuống (Runtime → Run all cũng được).
4. Tải 2 file .npz ở ô cuối về máy.

**DỪNG LẠI nếu ô kiểm tra GPU báo không có GPU.** Không chạy bằng CPU: lần đo trước trên CPU Colab mất khoảng 16 giờ cho 1 model. Đóng Colab lại rồi mở phiên mới để xin GPU khác.

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda.is_available():", torch.cuda.is_available())
assert torch.cuda.is_available(), "KHÔNG CÓ GPU — dừng tại đây, đừng chạy tiếp bằng CPU. Đóng phiên này và mở phiên Colab mới để xin GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("OK — có GPU, chạy tiếp.")

In [ ]:
%pip install -q sentence-transformers
print("cài xong sentence-transformers")

## 1. Nạp file đầu vào
Bấm **Choose Files** và chọn file `embedding-input-v7.3.2-filtered.jsonl` (12.910 dòng, mỗi dòng 1 chunk kèm trường `embed_text` đã ghép tiêu đề/mục/năm).

In [ ]:
from google.colab import files
import json, glob, os
uploaded = files.upload()
path = glob.glob("/content/embedding-input-*.jsonl")[0]
rows = [json.loads(l) for l in open(path, encoding="utf-8")]
print("số chunk:", len(rows))
print("mẫu embed_text:", rows[0]["embed_text"][:300].replace(chr(10), " | "))
ids = [r["chunk_id"] for r in rows]
texts = [r["embed_text"] for r in rows]

## 2. Embedding bằng bge-m3 (tạm chốt)
Model: `BAAI/bge-m3`, vector 1024 chiều, giới hạn 8.192 token (chunk dài nhất trong bộ khoảng 963 token nên không bị cắt cụt). Không cần thêm tiền tố gì vào văn bản.

In [ ]:
import time, numpy as np
from sentence_transformers import SentenceTransformer
t0 = time.time()
m1 = SentenceTransformer("BAAI/bge-m3", device="cuda")
emb1 = m1.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
print("bge-m3: shape", emb1.shape, "| thời gian: %.1f phút" % ((time.time()-t0)/60))
np.savez("vectors_bge_m3.npz", embeddings=emb1, chunk_ids=np.array(ids))
print("đã lưu vectors_bge_m3.npz")

## 3. Embedding bằng multilingual-e5-large (model so sánh, nhẹ hơn bge-m3)
Model: `intfloat/multilingual-e5-large`, vector 1024 chiều, giới hạn 512 token (khoảng 1,9% chunk sẽ bị cắt cụt — xem ô kiểm tra ở dưới). **Lưu ý:** model họ e5 bắt buộc thêm tiền tố `passage: ` trước văn bản cần embed, và `query: ` trước câu hỏi.

In [ ]:
t0 = time.time()
m2 = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
texts_e5 = ["passage: " + t for t in texts]
emb2 = m2.encode(texts_e5, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
print("e5-large: shape", emb2.shape, "| thời gian: %.1f phút" % ((time.time()-t0)/60))
np.savez("vectors_e5_large.npz", embeddings=emb2, chunk_ids=np.array(ids))
print("đã lưu vectors_e5_large.npz")

## 4. Kiểm tra cắt cụt (truncation)
Đếm số chunk dài hơn giới hạn token của từng model. Kỳ vọng: bge-m3 = 0 chunk, e5-large khoảng 264 chunk (1,9%).

In [ ]:
from transformers import AutoTokenizer
for name, limit in [("BAAI/bge-m3", 8192), ("intfloat/multilingual-e5-large", 512)]:
    tk = AutoTokenizer.from_pretrained(name)
    over = 0
    for t in texts[:2000]:
        if len(tk(t, add_special_tokens=True)["input_ids"]) > limit:
            over += 1
    print(f"{name}: {over}/2000 chunk mẫu vượt {limit} token")
print("Ghi chú: đếm trên 2.000 chunk mẫu để chạy nhanh; số đầy đủ tính lại sau nếu cần.")

## 5. Kiểm tra nhanh: truy xuất thử 3 câu hỏi
Embed 3 câu trong bộ eval v5 rồi tìm 3 chunk gần nhất theo cosine similarity cho từng model. Chỉ để kiểm tra vector chạy được, chưa phải chấm chính thức (chấm chính thức làm ở bước 5 trên cả 50 câu).

In [ ]:
import torch as _t
tests = [
    ("V5-A01", "Hạn nộp học phí học kỳ 1 năm học 2026-2027 là khi nào?"),
    ("V5-A07", "Học phí 1 tín chỉ ngành Dược học năm 2026 là bao nhiêu cho từng mức?"),
    ("V5-H03", "Mỗi học kỳ sinh viên phải đăng ký tối thiểu bao nhiêu tín chỉ và tối đa bao nhiêu tín chỉ?"),
]
for qid, q in tests:
    print("="*80)
    print(qid, "—", q)
    q1 = m1.encode([q], normalize_embeddings=True)
    q2 = m2.encode(["query: " + q], normalize_embeddings=True)
    for name, embs, qq in [("bge-m3", emb1, q1), ("e5-large", emb2, q2)]:
        sims = (embs @ qq[0]).argsort()[::-1][:3]
        print(f"  [{name}] top 3:")
        for i in sims:
            print(f"    {float(embs[i] @ qq[0]):.3f}  {ids[i]}")

## 6. Tải file vector về máy
2 file: `vectors_bge_m3.npz` và `vectors_e5_large.npz` (mỗi file khoảng 53 MB, gồm vector và chunk_id tương ứng). Giữ 2 file này để dùng ở bước 5.

In [ ]:
from google.colab import files as _f
_f.download("vectors_bge_m3.npz")
_f.download("vectors_e5_large.npz")
print("Xong bước 4: đã tải 2 file vector. Chụp lại thời gian chạy của từng model ở ô 2 và ô 3 để ghi vào báo cáo.")